In [ ]:
import pandas as pd

def extract_user_profile(temp_df: pd.DataFrame):
    # Define columns to extract
    columns_to_extract = [
        'user_id', 'article_id', 'pred',
        'presentation_type_formal_x', 'presentation_type_creative_x', 'presentation_type_students_x',
        'presentation_type_technical_x', 'presentation_type_branding_x',
        'audience_type_academic', 'audience_type_business',
        'audience_type_general', 'audience_type_specialized',
        'days_since_last_rating'
    ]
    
    # Extract relevant data
    extracted_df = temp_df[columns_to_extract]
    
    # Extract a sample user profile (first user in dataframe)
    first_user = extracted_df.iloc[0]
    
    # Identify the preferred presentation type
    preferred_presentation_type = next(
        (col.replace('presentation_type_', '').replace('_x', '') 
         for col in columns_to_extract[3:8] if first_user[col] == 1),
        None
    )
    
    # Identify the preferred audience type
    preferred_audience_type = next(
        (col.replace('audience_type_', '') 
         for col in columns_to_extract[8:12] if first_user[col] == 1),
        None
    )
    
    # Construct user profile dictionary
    user_profile = {
        'user_id': int(first_user['user_id']),
        'preferred_presentation_type': preferred_presentation_type,
        'audience_type': preferred_audience_type
    }
    
    return extracted_df, user_profile

# Example usage:
extracted_df, user_profile = extract_user_profile(temp_df.copy())
# print(extracted_df.head())
# print(user_profile)

def extract_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extracts specific columns from the given DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
    
    Returns:
        pd.DataFrame: A new DataFrame containing only the specified columns.
    """
    columns_to_extract = [
        'user_id',
        'article_id',
        'rating',
        'pred',
        'Boring presentation',
         'Graphics',
         'Readability',
         'Consistency',
         'Images',
         'Bullets',
         'Text size',
         'Text heavy',
         'Tables',
         'Agenda',
         'Infographics',
         'Positioning',
         'Presentation length',
         'General tips',
         'Presentation skills',
         'Powerpoint knowledge',
         'tips and tricks',
         'Explanation of the problem',
         'presentation_type_formal_y',
         'presentation_type_creative_y',
         'presentation_type_students_y',
         'presentation_type_technical_y',
         'presentation_type_branding_y',
         'audience_type_academic',
         'audience_type_business',
         'audience_type_kids',
         'audience_type_general',
         'audience_type_creative',
         'audience_type_specialized',
         'days_since_last_rating',
         'submission_date',
         'popularity'
    ]

    return df[columns_to_extract]

#extracted_df = extract_columns(temp_df.copy())

In [1]:
# Re-import necessary packages after kernel reset
import numpy as np
import pandas as pd
from datetime import datetime

# Step 1: Calculate freshness score from submission date 0normalized between 1 and 0
def compute_freshness_score(df, date_column='submission_date'):
    # Make sure submission_date is timezone-aware in UTC
    df[date_column] = pd.to_datetime(df[date_column], utc=True)
    
    # Get the current UTC time as a timezone-aware timestamp
    today = pd.Timestamp(datetime.now(), tz='UTC')
    
    # Compute age in days
    df['age_days'] = (today - df[date_column]).dt.days
    
    # Normalize freshness
    df['freshness_score'] = 1 - (df['age_days'] / df['age_days'].max())
    
    return df


# Step 2: Calculate inverse popularity -normalize between 0 and 1
def compute_inverse_popularity(df):
    df['inverse_popularity'] = 1 / (df['popularity'] + 0.01)
    df['inverse_popularity'] = (df['inverse_popularity'] - df['inverse_popularity'].min()) / (
        df['inverse_popularity'].max() - df['inverse_popularity'].min())
    return df

# Step 3: Adjust long-tail boost based on freshness and rarity 
def apply_long_tail_boost(df, gamma=0.5):
    df = compute_freshness_score(df)
    df = compute_inverse_popularity(df)
    df['long_tail_boost'] = gamma * df['freshness_score'] + (1 - gamma) * df['inverse_popularity']
    return df

# Step 4: Apply audience type boost
def compute_audience_boost(row, user_profile):
    audience_type = user_profile.get('audience_type')
    boost = 0
    if audience_type:
        for col in row.index:
            if col.startswith('audience_type_') and audience_type in col and row[col] == 1:
                boost = 1
                break
    return boost

# Final utility score computation
def finalize_recommendations(predicted_articles, user_profile, engagement_time,
                              alpha=0.2, beta=0.3, gamma=0.5):
    recs = predicted_articles.copy()

    # Preferred presentation sorting
    preferred_type = user_profile.get('preferred_presentation_type')
    if preferred_type in recs.columns:
        recs['is_preferred'] = recs[preferred_type] == 1
        recs = recs.sort_values(by='is_preferred', ascending=False).drop(columns='is_preferred')

    # Audience boost
    recs['audience_boost'] = recs.apply(lambda row: compute_audience_boost(row, user_profile), axis=1)

    # Long-tail boost (freshness + rarity)
    recs = apply_long_tail_boost(recs, gamma=gamma)

    # Final utility score using weighted strategy
    recs['utility_score'] = (
        recs['pred'] *
        (1 + alpha * recs['audience_boost']) *
        (1 + beta * recs['long_tail_boost'])
    )

    # Sort and limit
    recs = recs.sort_values('utility_score', ascending=False)
    recs = recs.drop_duplicates(subset='article_id', keep='first')
    max_recs = int(engagement_time / 2)
    return recs.head(max_recs)

def add_utility_scr_recommendations(temp_df, engagement_time=30, alpha=0.2, beta=0.3, gamma=0.5):
    """
    Generate final article recommendations for a user based on input DataFrame and engagement preferences.

    Parameters:
        temp_df (pd.DataFrame): Input dataset with article metadata.
        engagement_time (int): Time user is willing to engage (in minutes).
        alpha (float): Weight for audience-based boost.
        beta (float): Weight for long-tail (rarity) boost.
        gamma (float): Weight for freshness-rarity blend.

    Returns:
        final_recommendations (pd.DataFrame): Ranked and boosted article recommendations.
    """
    predicted_articles = extract_columns(temp_df.copy())
    extracted_df, user_profile = extract_user_profile(temp_df.copy())

    final_recommendations = finalize_recommendations(
        predicted_articles=predicted_articles,
        user_profile=user_profile,
        engagement_time=engagement_time,
        alpha=alpha,
        beta=beta,
        gamma=gamma
    )
    
    print("Final Recommendations (Top Items):")
    print(final_recommendations[['article_id','days_since_last_rating','pred','rating','utility_score']].head(10))

    return final_recommendations

######## EXAMPLE USAGE ###############

# recommendations = add_utility_scr_recommendations(temp_df, engagement_time=45, alpha=0.3, beta=0.2, gamma=0.5)

# print("Final Recommendations (Top Items):")
# print(final_recommendations[['article_id','days_since_last_rating','pred','rating','utility_score']].head(10))